# imports

In [1]:
import os
import random
from google.cloud import texttospeech_v1beta1 as texttospeech
#from google.cloud import storage
import os
from google.cloud import texttospeech
from datasets import load_from_disk

/home/peter/miniconda3/envs/thesis/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# TTS play

In [2]:
#from dotenv import load_dotenv
#load_dotenv() # probably unnecessary for future runs

In [3]:
PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT")

In [4]:
client = texttospeech.TextToSpeechClient()

In [5]:
voices = client.list_voices(language_code="en-US").voices
en_us_voices = [v.name for v in voices if "en-US" in v.language_codes]

just shuffle voices and increment over list to get even voice spread over dataset.

In [11]:
random.seed(42)  # seed once for reproducibility
shuffled_voices = random.sample(en_us_voices, len(en_us_voices))

def get_voice(idx):
    return shuffled_voices[idx % len(shuffled_voices)]

In [ ]:
from pathlib import Path

# Walk up from the notebook's location to find the project root
PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / ".git").exists(): # anchor project root to .git
    PROJECT_ROOT = PROJECT_ROOT.parent

In [ ]:
# Set data path
DATA_PATH = PROJECT_ROOT / "data" / "fleurs" / "synthetic"
# Set output path
output_dir = DATA_PATH / "generated"

# TTS call function

In [ ]:
def synthesize(text: str, output_filepath: str = "output.mp3"):
    """Synthesizes speech from the input text and saves it to an MP3 file.

    Args:
        prompt: Styling instructions on how to synthesize the content in
          the text field.
        text: The text to synthesize.
        output_filepath: The path to save the generated audio file.
          Defaults to "output.mp3".
    """
    client = texttospeech.TextToSpeechClient()

    synthesis_input = texttospeech.SynthesisInput(text=text)

    # Select the voice you want to use.
    voice = texttospeech.VoiceSelectionParams(
    language_code="en-US",
    name=random.choice(en_us_voices)
)

    audio_config = texttospeech.AudioConfig(
        audio_encoding=texttospeech.AudioEncoding.MP3
    )

    # Perform the text-to-speech request on the text input with the selected
    # voice parameters and audio file type.
    response = client.synthesize_speech(
        input=synthesis_input, voice=voice, audio_config=audio_config
    )

    # The response's audio_content is binary, so wb for write binary.
    with open(output_filepath, "wb") as out:
        out.write(response.audio_content)
        print(f"Audio content written to file: {output_filepath}")


In [ ]:
synthesize()

# load data

In [ ]:
ds = load_from_disk(DATA_PATH)

/home/peter/miniconda3/envs/thesis/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Parameter 'format_kwargs'={} of the transform datasets.arrow_dataset.Dataset.set_format couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


In [5]:
ds[0]

{'audio': {'path': 'carbad.wav',
  'array': array([0., 0., 0., ..., 0., 0., 0.], shape=(13440,)),
  'sampling_rate': 16000},
 'phonetic': 'k a ɾˠ ə bˠ ə d̪ˠ',
 'English ASR transcriptions': 'ʌ ɾ ə b ʌ d'}

# Batch synthesis

In [ ]:
from datasets import  Dataset
import time


ds = load_from_disk(str(PROJECT_ROOT / "data/fleurs/synthetic"))

output_dir = PROJECT_ROOT / "data/fleurs/synthetic_audio"
output_dir.mkdir(parents=True, exist_ok=True)

def synthesize_example(example, idx):
    ipa = example["English ASR transcriptions"]
    output_path = output_dir / f"{idx}.mp3"
    
    # skip if already synthesised (safe to re-run)
    if output_path.exists():
        example["synthetic_audio_path"] = str(output_path)
        return example

    ssml = f'<speak><phoneme alphabet="ipa" ph="{ipa}">word</phoneme></speak>'
    synthesis_input = texttospeech.SynthesisInput(ssml=ssml)

    voice = texttospeech.VoiceSelectionParams(
        language_code="en-US",
        name="en-US-Standard-C"
    )
    audio_config = texttospeech.AudioConfig(
        audio_encoding=texttospeech.AudioEncoding.MP3
    )

    response = client.synthesize_speech(
        input=synthesis_input, voice=voice, audio_config=audio_config
    )

    with open(output_path, "wb") as f:
        f.write(response.audio_content)

    time.sleep(0.1)  # avoid hammering the API
    example["synthetic_audio_path"] = str(output_path)
    return example

ds = ds.map(synthesize_example, with_indices=True, load_from_cache_file=False)
ds.save_to_disk(str(PROJECT_ROOT / "data/fleurs/synthetic"))
